# 02 - Adsorbate Chemistry and Reference Energies

This notebook moves from database cartography to chemistry.

We now ask:

1. Which adsorbates are represented?
2. Which gas-phase references are available in the same database?
3. Can we estimate chemically meaningful adsorption energies using
   clean-surface references contained in the dataset itself?

The analysis focuses on CO, CH3O, HCOO and COOH because those are abundant
enough to support comparative materials discussion.

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import subprocess
import sys
import tempfile

sys.path.insert(0, r"/Users/dk2994/Desktop/git/DFTDataFrame/src")
import DFTDataFrame.Tools as OP

pd = OP.pd
np = OP.np
plt = OP.plt

DATA_PATH = Path(r"/Users/dk2994/Desktop/Uni/scripts/created_frame.hdf")
DFTDATAFRAME_SRC = Path(r"/Users/dk2994/Desktop/git/DFTDataFrame/src")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12


def load_onepiece_hdf(path=DATA_PATH, key="df"):
    """Load the local OnePiece-style HDF table.

    This notebook uses the local DFTDataFrame package as the available
    OnePiece-compatible analysis layer. The actual HDF payload is read through
    the pandas namespace exposed by that package.
    """
    path = Path(path)
    try:
        return OP.pd.read_hdf(path, key=key).copy()
    except Exception as original_error:
        helper_python = Path("/Users/dk2994/opt/anaconda3/bin/python")
        if not helper_python.exists():
            raise original_error
        output = Path(tempfile.NamedTemporaryFile(delete=False, suffix=".pkl", prefix="created_frame_").name)
        script = """
from pathlib import Path
import sys
import numpy as np
import pandas as pd
try:
    import numpy.core as numpy_core
    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
    import ase.constraints  # noqa: F401
except Exception:
    pass
source = Path(sys.argv[1])
key = sys.argv[2]
target = Path(sys.argv[3])
pd.read_hdf(source, key=key).to_pickle(target)
"""
        completed = subprocess.run(
            [str(helper_python), "-c", script, str(path), key, str(output)],
            capture_output=True,
            text=True,
            check=False,
        )
        if completed.returncode != 0:
            detail = completed.stderr.strip() or completed.stdout.strip()
            raise RuntimeError(f"Could not read {path}. Helper reader failed: {detail}") from original_error
        return OP.pd.read_pickle(output).copy()


ADSORBATE_TOKENS = [
    "CH3OH", "CH3O", "HCOOH", "H2COOH", "HCOO", "COOH", "CO2", "HCO", "CO"
]


def guess_adsorbate(name):
    if not isinstance(name, str):
        return ""
    for token in sorted(ADSORBATE_TOKENS, key=len, reverse=True):
        if re.search(rf"(^|[-_%]){re.escape(token)}($|[-_%])", name):
            return token
    return ""


def guess_record_class(name, path=""):
    text = f"{name} {path}".lower()
    if "gasphases" in text:
        return "gas"
    if "copt" in text:
        return "copt"
    if "convergence" in text:
        return "convergence"
    if "bulk" in text:
        return "bulk"
    if "clean" in text:
        return "clean_surface"
    if guess_adsorbate(name):
        return "adsorbate"
    return "other"


def guess_facet(name):
    if not isinstance(name, str):
        return ""
    for facet in ["100", "110", "111", "211", "221"]:
        if facet in name:
            return facet
    return ""


def material_family(name):
    if not isinstance(name, str):
        return "unknown"
    token = name.split("-")[0]
    return token or "unknown"


def surface_key_from_name(name):
    if not isinstance(name, str):
        return ""
    key = name
    key = re.sub(r"-copt-.*$", "", key)
    key = re.sub(r"-(CH3OH|CH3O|HCOOH|H2COOH|HCOO|COOH|CO2|HCO|CO)([-_%].*)?$", "", key)
    key = re.sub(r"-[0-9]+$", "", key)
    return key


def add_taxonomy(df):
    out = df.copy()
    out["record_class"] = [guess_record_class(n, p) for n, p in zip(out["Name"], out["Path"], strict=False)]
    out["facet"] = out["Name"].map(guess_facet)
    out["material_family"] = out["Name"].map(material_family)
    out["adsorbate"] = out["Name"].map(guess_adsorbate)
    out["surface_key"] = out["Name"].map(surface_key_from_name)
    return out


def formula_counts(formula):
    counts = {}
    if not isinstance(formula, str):
        return counts
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


def add_formula_features(df):
    out = df.copy()
    parsed = out["Formula"].map(formula_counts)
    all_elements = sorted({el for counts in parsed for el in counts})
    for el in all_elements:
        out[el] = parsed.map(lambda counts: counts.get(el, 0))
    out["n_elements"] = parsed.map(len)
    out["n_atoms_formula"] = parsed.map(lambda counts: sum(counts.values()))
    return out


def gas_reference_energy(df, token):
    pattern = rf"^gasphases-{re.escape(token)}(?:$|[-_].*)"
    subset = df[df["Name"].astype(str).str.contains(pattern, regex=True, case=False, na=False)]
    subset = subset[pd.to_numeric(subset["E"], errors="coerce").notna()]
    if subset.empty:
        return np.nan
    return float(subset["E"].iloc[0])


def assign_clean_references(df):
    out = df.copy()
    clean = out[out["record_class"] == "clean_surface"].copy()
    clean = clean[pd.to_numeric(clean["E"], errors="coerce").notna()]
    clean_map = clean.groupby("surface_key")["E"].min().to_dict()
    clean_name_map = clean.sort_values("E").drop_duplicates("surface_key").set_index("surface_key")["Name"].to_dict()
    out["clean_ref_E"] = out["surface_key"].map(clean_map)
    out["clean_ref_name"] = out["surface_key"].map(clean_name_map)
    out["delta_E_surface"] = pd.to_numeric(out["E"], errors="coerce") - pd.to_numeric(out["clean_ref_E"], errors="coerce")
    return out


def adsorption_energy_conventions(df):
    out = df.copy()
    e_co = gas_reference_energy(out, "CO")
    e_h2 = gas_reference_energy(out, "H2")
    e_ch3oh = gas_reference_energy(out, "CH3OH")
    e_hcooh = gas_reference_energy(out, "HCOOH")
    out["E_ads_CO"] = np.where(out["adsorbate"].eq("CO"), out["E"] - out["clean_ref_E"] - e_co, np.nan)
    out["E_ads_CH3O_from_CH3OH"] = np.where(
        out["adsorbate"].eq("CH3O"),
        out["E"] - out["clean_ref_E"] - e_ch3oh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_HCOO_from_HCOOH"] = np.where(
        out["adsorbate"].eq("HCOO"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    out["E_ads_COOH_from_HCOOH"] = np.where(
        out["adsorbate"].eq("COOH"),
        out["E"] - out["clean_ref_E"] - e_hcooh + 0.5 * e_h2,
        np.nan,
    )
    return out


df_raw = load_onepiece_hdf()
df = add_formula_features(add_taxonomy(df_raw))
df["E"] = pd.to_numeric(df["E"], errors="coerce")
df["fmax"] = pd.to_numeric(df["fmax"], errors="coerce")
df["has_structure"] = df["struc"].map(lambda value: value.__class__.__name__ == "Atoms")
df["has_contcar"] = df["CONTCAR"].map(lambda value: value.__class__.__name__ == "Atoms")

## 1. Build the adsorption-analysis table

In [ ]:
analysis = adsorption_energy_conventions(assign_clean_references(df))
ads = analysis[analysis["record_class"].isin(["adsorbate", "copt"])].copy()
ads = ads[ads["clean_ref_E"].notna()].copy()
ads[["Name", "adsorbate", "clean_ref_name", "E", "clean_ref_E", "delta_E_surface"]].head(20)

## 2. Normalize obvious naming variants with the OnePiece helper

In [ ]:
ads = OP.unify_adsorbates(ads, "HCOOH", "COOH", "adsorbate")
ads = OP.unify_adsorbates(ads, "H2COOH", "COOH", "adsorbate")
ads["adsorbate"].value_counts().head(15)

## 3. Gas-phase reference energies present in the file

In [ ]:
gas_reference_table = pd.DataFrame({
    "species": ["CO", "H2", "CH3OH", "HCOOH"],
    "E_gas / eV": [
        gas_reference_energy(analysis, "CO"),
        gas_reference_energy(analysis, "H2"),
        gas_reference_energy(analysis, "CH3OH"),
        gas_reference_energy(analysis, "HCOOH"),
    ],
})
gas_reference_table

## 4. Which adsorbates are abundant enough for comparison?

In [ ]:
adsorbate_counts = ads["adsorbate"].replace("", np.nan).dropna().value_counts()
adsorbate_counts.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
adsorbate_counts.head(12).sort_values().plot(kind="barh", ax=ax, color="#54a24b")
ax.set_title("Most common adsorbates in the database")
ax.set_xlabel("rows")
ax.set_ylabel("adsorbate")
plt.tight_layout()
plt.show()

## 5. Adsorption energies by convention

In [ ]:
energy_columns = [
    "E_ads_CO",
    "E_ads_CH3O_from_CH3OH",
    "E_ads_HCOO_from_HCOOH",
    "E_ads_COOH_from_HCOOH",
]
ads[["Name", "adsorbate", "facet", "material_family", *energy_columns]].head(20)

In [ ]:
co_like = ads[ads["E_ads_CO"].notna()].copy()
ch3o_like = ads[ads["E_ads_CH3O_from_CH3OH"].notna()].copy()
hcoo_like = ads[ads["E_ads_HCOO_from_HCOOH"].notna()].copy()
cooh_like = ads[ads["E_ads_COOH_from_HCOOH"].notna()].copy()

summary = pd.DataFrame({
    "family": ["CO*", "CH3O*", "HCOO*", "COOH*"],
    "rows": [len(co_like), len(ch3o_like), len(hcoo_like), len(cooh_like)],
    "mean_energy": [
        co_like["E_ads_CO"].mean(),
        ch3o_like["E_ads_CH3O_from_CH3OH"].mean(),
        hcoo_like["E_ads_HCOO_from_HCOOH"].mean(),
        cooh_like["E_ads_COOH_from_HCOOH"].mean(),
    ],
})
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
plot_specs = [
    (co_like, "E_ads_CO", "CO adsorption"),
    (ch3o_like, "E_ads_CH3O_from_CH3OH", "CH3O adsorption from CH3OH + 1/2 H2"),
    (hcoo_like, "E_ads_HCOO_from_HCOOH", "HCOO adsorption from HCOOH - 1/2 H2"),
    (cooh_like, "E_ads_COOH_from_HCOOH", "COOH adsorption from HCOOH - 1/2 H2"),
]
for ax, (table, column, title) in zip(axes.ravel(), plot_specs, strict=False):
    values = pd.to_numeric(table[column], errors="coerce").dropna()
    ax.hist(values, bins=30, color="#4c78a8", alpha=0.85)
    ax.axvline(values.mean(), color="crimson", linestyle="--", linewidth=1.5, label=f"mean = {values.mean():.2f} eV")
    ax.set_title(title)
    ax.set_xlabel("adsorption energy / eV")
    ax.set_ylabel("count")
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Surface dependence of CO adsorption

In [ ]:
co_surface = (
    co_like.groupby(["material_family", "facet"])
    .agg(rows=("Name", "size"), mean_Eads=("E_ads_CO", "mean"), std_Eads=("E_ads_CO", "std"))
    .query("rows >= 3")
    .sort_values(["material_family", "facet"])
)
co_surface.head(30)

In [ ]:
plot_table = co_surface.reset_index()
labels = plot_table["material_family"] + "-" + plot_table["facet"].replace("", "na")
fig, ax = plt.subplots(figsize=(12, 6))
ax.errorbar(labels, plot_table["mean_Eads"], yerr=plot_table["std_Eads"].fillna(0), fmt="o", color="#e45756")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("CO adsorption energy across material families and facets")
ax.set_ylabel("E_ads(CO) / eV")
ax.set_xlabel("surface family")
plt.xticks(rotation=70, ha="right")
plt.tight_layout()
plt.show()

The chemical value of this notebook is not that every convention is unique or
final, but that the database itself already contains the reference information
needed to formulate adsorption hypotheses. That is exactly what a thesis-scale
materials workflow needs: transparent conventions tied to explicit rows.